[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/099.%20The%20Electric%20Vehicle%20Routing%20Problem%20%28EVRP%29/P99-Tier-1_executed.ipynb)

---


# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 1: Mixed-Integer Programming Formulation

### Purpose and inputs

This notebook formulates and solves a small electric vehicle routing problem as a mixed-integer linear program (MILP). The input is a set of depot, customer, and charging-station locations; a fleet of electric vehicles; and battery and energy-consumption data. The model chooses vehicle routes and charging amounts. Its primary output is a minimum-distance set of routes that serves every customer once and respects the modeled battery constraints.

A mixed-integer program combines **binary decisions**—here, whether a vehicle uses a particular arc—with **continuous quantities**, such as a vehicle's battery level. The solver searches over these decisions while enforcing the mathematical constraints below.

### Model assumptions and scope

- Node 0 is the single depot. Each vehicle may leave it at most once and must return if it is used.
- Every customer is visited exactly once. Each charging station may be visited at most once by each vehicle; this matches the node-indexed battery and charging variables used in this formulation.
- Travel is represented by straight-line Euclidean distance in a two-dimensional plane. Energy use is proportional to distance, with the same consumption rate on every arc.
- Battery level is measured on arrival at a node. At a charging station, a vehicle may add energy before taking its next arc. Battery capacity cannot be exceeded, and the energy needed for every selected arc, including the return to the depot, must be available.
- Charging duration, customer service duration, time windows, traffic, charging losses, and vehicle-specific differences in travel or energy use are outside this model. The instance stores a charging rate and service times, but they do not enter the optimization because the objective and constraints contain no time variables.
- The objective minimizes total travel distance, not electricity cost, charging time, or the number of vehicles used.

### Sets, variables, and mathematical model

Let \(N\) be the set of nodes, \(C \subseteq N\) the customers, \(F \subseteq N\) the charging stations, and \(V\) the vehicles. The depot is node 0. For distinct nodes \(i,j \in N\), let \(d_{ij}\) be their Euclidean distance and \(e_{ij}=r d_{ij}\) the energy required to travel between them, where \(r\) is the energy-consumption rate.

The binary variable \(x_{ijk}\) equals 1 when vehicle \(k\) travels directly from node \(i\) to node \(j\), and 0 otherwise. The continuous variable \(u_{ik}\) is the battery energy of vehicle \(k\) on arrival at node \(i\). For \(i \in F\), \(y_{ik}\) is the energy added at station \(i\). A nonnegative auxiliary variable \(g_{ijk}\) carries connectivity flow along selected arcs; it ensures that customer visits belong to routes connected to the depot rather than disconnected cycles.

The model minimizes total distance:

$$
\min \quad \sum_{k \in V} \sum_{i \in N} \sum_{\substack{j \in N \\ j \ne i}} d_{ij}x_{ijk}.
$$

Each customer has exactly one incoming route arc across the fleet. At customers and charging stations, each vehicle has equal numbers of incoming and outgoing arcs. Each vehicle leaves the depot at most once, and its number of returns equals its number of departures:

$$
\sum_{k \in V} \sum_{\substack{j \in N \\ j \ne i}}x_{jik}=1 \quad (i \in C),
$$

$$
\sum_{\substack{j \in N \\ j \ne i}}x_{jik}
=
\sum_{\substack{j \in N \\ j \ne i}}x_{ijk}
\quad (i \in C \cup F,\ k \in V),
$$

$$
\sum_{\substack{j \in N \\ j \ne 0}}x_{0jk}\leq 1,
\qquad
\sum_{\substack{i \in N \\ i \ne 0}}x_{i0k}
=
\sum_{\substack{j \in N \\ j \ne 0}}x_{0jk}
\quad (k \in V).
$$

For a selected arc from \(i\) to a non-depot node \(j\), the arrival battery equals the departure battery after any charging at \(i\), minus arc energy use. A sufficiently large constant \(M_k\) relaxes these two inequalities when the arc is not selected. The return arc is handled separately because the depot's battery variable represents its initial charge, not a new battery state after a completed route:

$$
u_{jk} \leq u_{ik}+y_{ik}-e_{ij}+M_k(1-x_{ijk}),
\qquad (i\ne j,\ j\ne 0,\ k\in V),
$$

$$
u_{jk} \geq u_{ik}+y_{ik}-e_{ij}-M_k(1-x_{ijk}),
\qquad (i\ne j,\ j\ne 0,\ k\in V),
$$

$$
u_{ik}+y_{ik} \geq e_{i0}-M_k(1-x_{i0k}),
\qquad (i\ne 0,\ k\in V).
$$

Here, \(y_{ik}=0\) when \(i\) is not a charging station; in the code, charging variables are defined only for stations. The remaining battery bounds and initial conditions are

$$
0\leq u_{ik}\leq B_k,\qquad
0\leq y_{ik}\leq B_k-u_{ik}\ (i\in F),\qquad
u_{0k}=q_k,
$$

where \(B_k\) is vehicle \(k\)'s battery capacity and \(q_k\) is its initial charge. Charging stations are limited to one visit per vehicle:

$$
\sum_{\substack{j \in N \\ j \ne i}}x_{jik}\leq 1
\quad (i\in F,\ k\in V).
$$

The connectivity-flow constraints use one unit of flow for each customer served by a vehicle. Flow is limited to selected arcs, each visited customer consumes one unit, charging stations conserve flow, and the depot supplies the units needed by that vehicle's customers. These constraints prevent customer-only or station-mediated customer subtours that are disconnected from the depot:

$$
0\leq g_{ijk}\leq |C|x_{ijk},
$$

$$
\sum_{\substack{j \in N \\ j \ne i}}g_{jik}
-
\sum_{\substack{j \in N \\ j \ne i}}g_{ijk}
=
\sum_{\substack{j \in N \\ j \ne i}}x_{jik}
\quad (i\in C),
$$

$$
\sum_{\substack{j \in N \\ j \ne i}}g_{jik}
-
\sum_{\substack{j \in N \\ j \ne i}}g_{ijk}=0
\quad (i\in F),
$$

$$
\sum_{\substack{j \in N \\ j \ne 0}}g_{0jk}
-
\sum_{\substack{i \in N \\ i \ne 0}}g_{i0k}
=
\sum_{i\in C}\sum_{\substack{j \in N \\ j \ne i}}x_{jik}
\quad (k\in V).
$$

The variable domains are \(x_{ijk}\in\{0,1\}\) and \(u_{ik},y_{ik},g_{ijk}\geq 0\), for their respective index sets. In the implementation, the distance and energy matrices supply \(d_{ij}\) and \(e_{ij}\); `x`, `u`, `y`, and `g` implement the decision variables; and the loops in `solve_evrp_mip` add these constraints. The number of binary routing variables grows on the order of \(|N|^2|V|\). MILP solution time can grow rapidly with instance size; worst-case complexity is exponential, although actual performance depends on the instance and solver.

### Solution sequence

1. Define the nodes, vehicles, and physical parameters.
2. Calculate Euclidean distances and the corresponding arc energy requirements.
3. Build the MILP and solve it with PuLP's CBC solver.
4. Extract routes, arrival battery levels, and charging actions from an optimal solution.
5. Visualize and independently check the extracted solution.
6. Solve the same model for a range of battery capacities as a sensitivity analysis.

### Concrete example

The example contains one depot, three customers, two charging stations, and two electric vehicles. Each vehicle has a battery capacity and initial charge of 100 kWh; energy consumption is 0.8 kWh per kilometre, and the stated charging rate is 50 kW. Coordinates are interpreted in kilometres. The charging rate and customer service times are recorded as instance data but do not affect this distance-and-battery model.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pulp
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Node:
    """Store a node's location and role in the EVRP network."""

    id: int
    x: float
    y: float
    node_type: str
    service_time: float = 0.0


@dataclass
class Vehicle:
    """Store the battery data for one electric vehicle."""

    id: int
    battery_capacity: float
    initial_charge: float


@dataclass
class EVRPInstance:
    """Collect the data defining one EVRP instance."""

    nodes: List[Node]
    vehicles: List[Vehicle]
    energy_consumption_rate: float
    charging_rate: float


def create_example_instance() -> EVRPInstance:
    """Create the example instance described in the problem statement."""
    nodes = [
        Node(0, 0, 0, "depot"),
        Node(1, 10, 5, "customer", service_time=0.5),
        Node(2, 8, 12, "customer", service_time=0.5),
        Node(3, 15, 8, "customer", service_time=0.5),
        Node(4, 5, 8, "charging_station"),
        Node(5, 12, 2, "charging_station"),
    ]

    vehicles = [
        Vehicle(0, battery_capacity=100, initial_charge=100),
        Vehicle(1, battery_capacity=100, initial_charge=100),
    ]

    return EVRPInstance(
        nodes=nodes,
        vehicles=vehicles,
        energy_consumption_rate=0.8,
        charging_rate=50.0,
    )


instance = create_example_instance()
print(
    f"Created an EVRP instance with {len(instance.nodes)} nodes "
    f"and {len(instance.vehicles)} vehicles."
)

In [ ]:
def calculate_distance(node1: Node, node2: Node) -> float:
    """Return the Euclidean distance between two node coordinates."""
    return float(np.hypot(node1.x - node2.x, node1.y - node2.y))


def calculate_energy_consumption(distance: float, rate: float) -> float:
    """Return the energy needed to travel a distance at a fixed rate."""
    return distance * rate


def build_distance_and_energy_matrices(
    instance: EVRPInstance,
) -> Tuple[np.ndarray, np.ndarray]:
    """Build arc data from the locations and consumption rate in an instance."""
    n_nodes = len(instance.nodes)
    distances = np.zeros((n_nodes, n_nodes), dtype=float)
    energy_consumption = np.zeros((n_nodes, n_nodes), dtype=float)

    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                distances[i, j] = calculate_distance(
                    instance.nodes[i], instance.nodes[j]
                )
                energy_consumption[i, j] = calculate_energy_consumption(
                    distances[i, j], instance.energy_consumption_rate
                )

    return distances, energy_consumption


distances, energy_consumption = build_distance_and_energy_matrices(instance)

print("Distance matrix (km):")
print(np.round(distances, 2))
print("\nArc energy requirements (kWh):")
print(np.round(energy_consumption, 2))

In [ ]:
def solve_evrp_mip(instance: EVRPInstance):
    """Build and solve the EVRP mixed-integer programming model."""
    model = pulp.LpProblem("EVRP_MIP", pulp.LpMinimize)
    distances, energy_consumption = build_distance_and_energy_matrices(instance)

    N = range(len(instance.nodes))
    C = [i for i, node in enumerate(instance.nodes) if node.node_type == "customer"]
    F = [
        i
        for i, node in enumerate(instance.nodes)
        if node.node_type == "charging_station"
    ]
    V = range(len(instance.vehicles))
    depot_nodes = [i for i, node in enumerate(instance.nodes) if node.node_type == "depot"]
    if depot_nodes != [0]:
        raise ValueError("The model requires node 0 to be the single depot.")
    depot = 0

    # Explicit variable construction supports PuLP versions that do not expose dicts().
    x = {
        (i, j, k): pulp.LpVariable(f"x_{i}_{j}_{k}", cat="Binary")
        for i in N
        for j in N
        for k in V
        if i != j
    }
    u = {
        (i, k): pulp.LpVariable(f"u_{i}_{k}", lowBound=0, cat="Continuous")
        for i in N
        for k in V
    }
    y = {
        (i, k): pulp.LpVariable(f"y_{i}_{k}", lowBound=0, cat="Continuous")
        for i in F
        for k in V
    }
    g = {
        (i, j, k): pulp.LpVariable(f"g_{i}_{j}_{k}", lowBound=0, cat="Continuous")
        for i in N
        for j in N
        for k in V
        if i != j
    }

    model += pulp.lpSum(
        distances[i, j] * x[i, j, k]
        for i in N
        for j in N
        for k in V
        if i != j
    ), "Total_Travel_Distance"

    # Exactly one incoming route arc assigns each customer to one vehicle.
    for i in C:
        model += (
            pulp.lpSum(
                x[j, i, k] for j in N for k in V if j != i
            )
            == 1,
            f"Visit_Customer_{i}_Once",
        )

    # Flow conservation links arrivals and departures at every visited service node.
    for i in C + F:
        for k in V:
            model += (
                pulp.lpSum(x[j, i, k] for j in N if j != i)
                == pulp.lpSum(x[i, j, k] for j in N if j != i),
                f"Route_Flow_Node_{i}_Vehicle_{k}",
            )

    # A station has one node-indexed battery state, so a vehicle visits it at most once.
    for i in F:
        for k in V:
            model += (
                pulp.lpSum(x[j, i, k] for j in N if j != i) <= 1,
                f"Station_Visit_Limit_{i}_Vehicle_{k}",
            )

    for k in V:
        model += (
            pulp.lpSum(x[depot, j, k] for j in N if j != depot) <= 1,
            f"At_Most_One_Departure_Vehicle_{k}",
        )
        model += (
            pulp.lpSum(x[i, depot, k] for i in N if i != depot)
            == pulp.lpSum(x[depot, j, k] for j in N if j != depot),
            f"Return_To_Depot_Vehicle_{k}",
        )

    # The big-M value relaxes battery-transition bounds only when an arc is unused.
    for k in V:
        capacity = instance.vehicles[k].battery_capacity
        max_arc_energy = float(np.max(energy_consumption))
        big_m = max(2 * capacity, capacity + max_arc_energy)

        for i in N:
            model += (
                u[i, k] <= capacity,
                f"Battery_Capacity_Node_{i}_Vehicle_{k}",
            )

        model += (
            u[depot, k] == instance.vehicles[k].initial_charge,
            f"Initial_Battery_Vehicle_{k}",
        )

        for i in F:
            model += (
                y[i, k] <= capacity - u[i, k],
                f"Charging_Capacity_Station_{i}_Vehicle_{k}",
            )

        for i in N:
            for j in N:
                if i == j:
                    continue
                charge_at_origin = y[i, k] if i in F else 0

                if j != depot:
                    model += (
                        u[j, k]
                        <= u[i, k]
                        + charge_at_origin
                        - energy_consumption[i, j]
                        + big_m * (1 - x[i, j, k]),
                        f"Battery_Upper_{i}_{j}_Vehicle_{k}",
                    )
                    model += (
                        u[j, k]
                        >= u[i, k]
                        + charge_at_origin
                        - energy_consumption[i, j]
                        - big_m * (1 - x[i, j, k]),
                        f"Battery_Lower_{i}_{j}_Vehicle_{k}",
                    )
                else:
                    model += (
                        u[i, k]
                        + charge_at_origin
                        >= energy_consumption[i, depot]
                        - big_m * (1 - x[i, depot, k]),
                        f"Battery_Return_{i}_Vehicle_{k}",
                    )

    # Connectivity flow makes every served customer reachable from the depot.
    for i in N:
        for j in N:
            if i != j:
                for k in V:
                    model += (
                        g[i, j, k] <= len(C) * x[i, j, k],
                        f"Connectivity_Arc_{i}_{j}_Vehicle_{k}",
                    )

    for i in C:
        for k in V:
            model += (
                pulp.lpSum(g[j, i, k] for j in N if j != i)
                - pulp.lpSum(g[i, j, k] for j in N if j != i)
                == pulp.lpSum(x[j, i, k] for j in N if j != i),
                f"Connectivity_Customer_{i}_Vehicle_{k}",
            )

    for i in F:
        for k in V:
            model += (
                pulp.lpSum(g[j, i, k] for j in N if j != i)
                == pulp.lpSum(g[i, j, k] for j in N if j != i),
                f"Connectivity_Station_{i}_Vehicle_{k}",
            )

    for k in V:
        model += (
            pulp.lpSum(g[depot, j, k] for j in N if j != depot)
            - pulp.lpSum(g[i, depot, k] for i in N if i != depot)
            == pulp.lpSum(
                x[j, i, k] for i in C for j in N if j != i
            ),
            f"Connectivity_Depot_Vehicle_{k}",
        )

    print("Solving the mixed-integer programming model...")
    model.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=60))

    return model, x, u, y


model, x_vars, u_vars, y_vars = solve_evrp_mip(instance)
status = pulp.LpStatus[model.status]
print(f"\nOptimization status: {status}")

if status == "Optimal":
    print(f"Minimum total distance: {pulp.value(model.objective):.2f} km")
else:
    print("An optimal solution was not certified; route results are not reported as optimal.")

In [ ]:
def extract_solution(x_vars, u_vars, y_vars, instance):
    """Extract routes and their node-indexed battery and charging decisions."""
    solution = {
        "routes": [],
        "route_vehicle_ids": [],
        "total_distance": 0.0,
        "battery_levels": {},
        "charging_actions": {},
    }
    distances, _ = build_distance_and_energy_matrices(instance)
    tolerance = 1e-6

    for k in range(len(instance.vehicles)):
        route = []
        current = 0
        visited_arcs = set()

        while True:
            next_nodes = [
                j
                for j in range(len(instance.nodes))
                if j != current
                and pulp.value(x_vars[current, j, k]) is not None
                and pulp.value(x_vars[current, j, k]) > 0.5
            ]

            if not next_nodes:
                break
            if len(next_nodes) > 1:
                raise RuntimeError(
                    f"Vehicle {k} has multiple selected departures from node {current}."
                )

            next_node = next_nodes[0]
            arc = (current, next_node)
            if arc in visited_arcs:
                raise RuntimeError(f"Vehicle {k}'s selected route contains a cycle.")

            route.append(arc)
            visited_arcs.add(arc)
            current = next_node

            if current == 0:
                break

        if route:
            solution["routes"].append(route)
            solution["route_vehicle_ids"].append(k)

            visited_nodes = {0}
            for i, j in route:
                visited_nodes.add(i)
                visited_nodes.add(j)
                solution["total_distance"] += distances[i, j]

            for i in visited_nodes:
                battery = pulp.value(u_vars[i, k])
                if battery is not None:
                    solution["battery_levels"][(i, k)] = float(battery)

            station_nodes = {
                i
                for i in visited_nodes
                if instance.nodes[i].node_type == "charging_station"
            }
            for i in station_nodes:
                charge = pulp.value(y_vars[i, k])
                if charge is not None and charge > tolerance:
                    solution["charging_actions"][(i, k)] = float(charge)

    return solution


if status == "Optimal":
    solution = extract_solution(x_vars, u_vars, y_vars, instance)
else:
    solution = {
        "routes": [],
        "route_vehicle_ids": [],
        "total_distance": 0.0,
        "battery_levels": {},
        "charging_actions": {},
    }

print("Solution summary:")
print(f"Total distance: {solution['total_distance']:.2f} km")
print(f"Routes used: {len(solution['routes'])}")
print(f"Positive charging actions: {len(solution['charging_actions'])}")

for route_number, route in enumerate(solution["routes"], start=1):
    vehicle_id = solution["route_vehicle_ids"][route_number - 1]
    route_description = ["Depot"]
    for _, node_index in route:
        node = instance.nodes[node_index]
        if node.node_type == "customer":
            route_description.append(f"Customer {node_index}")
        elif node.node_type == "charging_station":
            route_description.append(f"Charging station {node_index}")
        else:
            route_description.append("Depot")
    print(
        f"\nRoute {route_number} (vehicle {vehicle_id}): "
        + " -> ".join(route_description)
    )

In [ ]:
def visualize_solution(instance, solution):
    """Plot routes, arrival battery levels, route distances, and instance data."""
    distances, _ = build_distance_and_energy_matrices(instance)
    route_colors = ["tab:blue", "tab:red", "tab:green", "tab:orange"]
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Electric Vehicle Routing Problem: MILP Solution",
        fontsize=16,
        fontweight="bold",
    )

    # Distinct labels make the node roles interpretable in the route legend.
    for i, node in enumerate(instance.nodes):
        if node.node_type == "depot":
            ax1.scatter(
                node.x, node.y, s=260, c="black", marker="s", label="Depot", zorder=5
            )
            label = "Depot"
        elif node.node_type == "customer":
            ax1.scatter(
                node.x,
                node.y,
                s=170,
                c="tab:blue",
                marker="o",
                label="Customer" if i == next(
                    index
                    for index, item in enumerate(instance.nodes)
                    if item.node_type == "customer"
                ) else None,
                zorder=4,
            )
            label = f"C{i}"
        else:
            ax1.scatter(
                node.x,
                node.y,
                s=190,
                c="tab:green",
                marker="^",
                label="Charging station" if i == next(
                    index
                    for index, item in enumerate(instance.nodes)
                    if item.node_type == "charging_station"
                ) else None,
                zorder=4,
            )
            label = f"CS{i}"
        ax1.annotate(
            label,
            (node.x, node.y),
            xytext=(5, 5),
            textcoords="offset points",
        )

    for route_index, route in enumerate(solution["routes"]):
        color = route_colors[route_index % len(route_colors)]
        vehicle_id = solution["route_vehicle_ids"][route_index]
        for i, j in route:
            ax1.plot(
                [instance.nodes[i].x, instance.nodes[j].x],
                [instance.nodes[i].y, instance.nodes[j].y],
                color=color,
                linewidth=2,
                alpha=0.75,
                label=f"Vehicle {vehicle_id} route" if i == route[0][0] and j == route[0][1] else None,
            )

    ax1.set_xlabel("X coordinate (km)")
    ax1.set_ylabel("Y coordinate (km)")
    ax1.set_title("Vehicle routes")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    for route_index, route in enumerate(solution["routes"]):
        vehicle_id = solution["route_vehicle_ids"][route_index]
        route_nodes = [route[0][0]] + [j for _, j in route]
        battery_levels = [
            solution["battery_levels"].get((node_index, vehicle_id), np.nan)
            for node_index in route_nodes
        ]
        ax2.plot(
            range(len(route_nodes)),
            battery_levels,
            marker="o",
            linewidth=2,
            label=f"Vehicle {vehicle_id}",
        )

    ax2.set_xlabel("Route position (arrival node)")
    ax2.set_ylabel("Arrival battery level (kWh)")
    ax2.set_title("Battery level at route nodes")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    route_distances = [
        sum(distances[i, j] for i, j in route)
        for route in solution["routes"]
    ]
    route_labels = [
        f"Vehicle {vehicle_id}"
        for vehicle_id in solution["route_vehicle_ids"]
    ]
    bars = ax3.bar(route_labels, route_distances, color=route_colors[:len(route_distances)])
    ax3.set_ylabel("Distance (km)")
    ax3.set_title("Distance by vehicle route")
    ax3.grid(True, alpha=0.3, axis="y")
    for bar, distance in zip(bars, route_distances):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{distance:.1f} km",
            ha="center",
            va="bottom",
        )

    total_traction_energy = (
        solution["total_distance"] * instance.energy_consumption_rate
    )
    stats_text = (
        "Solution statistics\n"
        "-------------------\n"
        f"Total distance: {solution['total_distance']:.2f} km\n"
        f"Route traction energy: {total_traction_energy:.2f} kWh\n"
        f"Routes used: {len(solution['routes'])}\n"
        f"Positive charging actions: {len(solution['charging_actions'])}\n"
        f"Customers: {sum(node.node_type == 'customer' for node in instance.nodes)}\n"
        "\nVehicle and energy data\n"
        "----------------------\n"
        f"Battery capacity: {instance.vehicles[0].battery_capacity} kWh\n"
        f"Energy use: {instance.energy_consumption_rate} kWh/km\n"
        f"Charging rate: {instance.charging_rate} kW"
    )
    ax4.text(
        0.05,
        0.5,
        stats_text,
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="center",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightgray", "alpha": 0.8},
    )
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis("off")
    ax4.set_title("Instance and solution summary")

    plt.tight_layout()
    plt.show()


visualize_solution(instance, solution)

In [ ]:
def verify_feasibility(instance, solution):
    """Check customer coverage, route endpoints, and battery use on each arc."""
    tolerance = 1e-5
    expected_customers = {
        i for i, node in enumerate(instance.nodes) if node.node_type == "customer"
    }
    customer_visits = {i: 0 for i in expected_customers}

    for route in solution["routes"]:
        for _, j in route:
            if j in customer_visits:
                customer_visits[j] += 1

    customers_served_once = all(count == 1 for count in customer_visits.values())
    battery_violations = 0
    route_endpoint_violations = 0
    charging_violations = 0

    for route_index, route in enumerate(solution["routes"]):
        vehicle_id = solution["route_vehicle_ids"][route_index]
        if not route or route[0][0] != 0 or route[-1][1] != 0:
            route_endpoint_violations += 1

        for i, j in route:
            if instance.nodes[i].node_type == "depot":
                available_energy = instance.vehicles[vehicle_id].initial_charge
            else:
                arrival_energy = solution["battery_levels"].get((i, vehicle_id))
                if arrival_energy is None:
                    battery_violations += 1
                    continue
                available_energy = arrival_energy

            charge = solution["charging_actions"].get((i, vehicle_id), 0.0)
            if charge > 0 and instance.nodes[i].node_type != "charging_station":
                charging_violations += 1

            capacity = instance.vehicles[vehicle_id].battery_capacity
            if available_energy + charge > capacity + tolerance:
                battery_violations += 1
            energy_needed = calculate_energy_consumption(
                calculate_distance(instance.nodes[i], instance.nodes[j]),
                instance.energy_consumption_rate,
            )
            if available_energy + charge + tolerance < energy_needed:
                battery_violations += 1

            if j != 0:
                next_arrival_energy = solution["battery_levels"].get((j, vehicle_id))
                if next_arrival_energy is None or not np.isclose(
                    next_arrival_energy,
                    available_energy + charge - energy_needed,
                    atol=tolerance,
                ):
                    battery_violations += 1

    stations_only = all(
        instance.nodes[i].node_type == "charging_station"
        for i, _ in solution["charging_actions"]
    )
    if not stations_only:
        charging_violations += 1

    print("Feasibility verification:")
    print(f"Every customer is visited exactly once: {customers_served_once}")
    print(f"Battery constraints are respected: {battery_violations == 0}")
    print(f"Charging occurs only at stations: {charging_violations == 0}")
    print(f"Routes start and end at the depot: {route_endpoint_violations == 0}")

    is_feasible = (
        customers_served_once
        and battery_violations == 0
        and charging_violations == 0
        and route_endpoint_violations == 0
    )
    print(f"\nOverall solution feasible: {is_feasible}")
    return is_feasible


feasible = verify_feasibility(instance, solution)

In [ ]:
def sensitivity_analysis(instance):
    """Re-solve the same MILP for the specified battery capacities."""
    print("Sensitivity analysis: battery capacity")
    print("=" * 58)

    battery_capacities = [60, 80, 100, 120, 140]
    results = []

    for capacity in battery_capacities:
        modified_instance = EVRPInstance(
            nodes=instance.nodes,
            vehicles=[
                Vehicle(vehicle.id, capacity, capacity)
                for vehicle in instance.vehicles
            ],
            energy_consumption_rate=instance.energy_consumption_rate,
            charging_rate=instance.charging_rate,
        )

        try:
            model, x_vars, u_vars, y_vars = solve_evrp_mip(modified_instance)
            solve_status = pulp.LpStatus[model.status]

            if solve_status == "Optimal":
                result_solution = extract_solution(
                    x_vars, u_vars, y_vars, modified_instance
                )
                results.append(
                    {
                        "battery_capacity": capacity,
                        "total_distance": result_solution["total_distance"],
                        "charging_stops": len(result_solution["charging_actions"]),
                        "status": solve_status,
                    }
                )
            else:
                results.append(
                    {
                        "battery_capacity": capacity,
                        "total_distance": None,
                        "charging_stops": None,
                        "status": solve_status,
                    }
                )
        except (RuntimeError, ValueError, pulp.PulpError) as error:
            results.append(
                {
                    "battery_capacity": capacity,
                    "total_distance": None,
                    "charging_stops": None,
                    "status": f"Error: {error}",
                }
            )

    print(
        f"{'Battery capacity (kWh)':<25} "
        f"{'Distance (km)':<18} {'Charging actions':<20} Status"
    )
    print("-" * 85)
    for result in results:
        distance = (
            f"{result['total_distance']:.2f}"
            if result["total_distance"] is not None
            else "N/A"
        )
        charging_actions = (
            str(result["charging_stops"])
            if result["charging_stops"] is not None
            else "N/A"
        )
        print(
            f"{result['battery_capacity']:<25} "
            f"{distance:<18} {charging_actions:<20} {result['status']}"
        )

    return results


sensitivity_results = sensitivity_analysis(instance)

## Role and limitations of the MILP tier

For an instance that the solver finishes to optimality, the MILP provides a minimum-distance solution for the stated mathematical model. Optimality is relative to the assumptions and constraints in this notebook; it does not imply that the routes are best under omitted considerations such as time windows, charging duration, traffic, or electricity prices. A time limit may prevent the solver from certifying optimality, so the reported status must be considered alongside any objective value.

### Strengths

- **Precise formulation:** routing, battery transitions, charging decisions, and depot connectivity are represented by explicit variables and constraints.
- **Optimality certification:** when the solver reports `Optimal`, the objective is optimal for this finite model, subject to solver tolerances.
- **Benchmarking:** small instances can provide exact reference solutions for evaluating other approaches.
- **Transparent feasibility checks:** extracted routes can be checked against customer coverage, depot returns, and battery use.

### Limitations

- **Scaling:** the number of routing decisions grows quadratically with the number of nodes and linearly with the number of vehicles; MILP solution time can grow rapidly as instances grow.
- **Model scope:** the result reflects Euclidean travel, fixed energy use per kilometre, and the battery and charging assumptions stated above. Time-related constraints are not represented.
- **Solver dependence:** available solvers, their settings, and the time limit affect whether optimality can be certified within a run.

| Aspect | Strength | Limitation |
|---|---|---|
| Solution quality | An optimal solution can be certified for the modeled instance. | Certification may require substantial computation as the instance grows. |
| Mathematical model | Decisions and constraints have explicit definitions. | Results are only as representative as the included assumptions. |
| Feasibility | Battery and route constraints are enforced and checked. | Time windows, charging duration, and other omitted constraints are not checked. |
| Implementation | PuLP provides a Python interface to the CBC solver used here. | Solver availability and settings influence execution and certification. |

The battery-capacity sensitivity analysis illustrates how changing one parameter affects model feasibility, distance, and charging decisions. Each capacity is evaluated using the same MILP formulation; the results should be interpreted using the solver status reported for that run.